# EDA and Visualization

Features computed. Now let's actually look at them.

The questions we care about: do chosen and rejected responses separate cleanly in feature space? Which features carry the most signal? Are there natural clusters in the embedding space before we even try to cluster formally? And does the helpful-base vs harmless-base split introduce any distributional quirks we should know about?

The answers here will directly inform what we expect from the models in Week 3.

In [ ]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from inference_lens.utils.logging import setup_logging

setup_logging()
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

import os
os.makedirs('../../reports/figures', exist_ok=True)

## Load features

In [ ]:
df = pd.read_parquet('../../data/processed/features_with_labels.parquet')
embeddings = np.load('../../data/processed/embeddings.npy')

print(f'Features: {df.shape}')
print(f'Embeddings: {embeddings.shape}')
df.head(3)

## Feature distributions: chosen vs rejected

Side-by-side KDE plots for each feature. The more the distributions diverge, the more useful that feature is to a classifier.

In [ ]:
feature_cols = ['token_length', 'type_token_ratio', 'flesch_score', 'rouge_l']
feature_labels = ['Token Length', 'Type-Token Ratio', 'Flesch Score', 'ROUGE-L']

chosen = df[df['label'] == 1]
rejected = df[df['label'] == 0]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (col, label) in enumerate(zip(feature_cols, feature_labels)):
    ax = axes[i]
    # clip extreme outliers for cleaner plots
    lo, hi = df[col].quantile(0.01), df[col].quantile(0.99)
    c_vals = chosen[col].clip(lo, hi)
    r_vals = rejected[col].clip(lo, hi)

    sns.kdeplot(c_vals, ax=ax, label='Chosen', fill=True, alpha=0.4, color='steelblue')
    sns.kdeplot(r_vals, ax=ax, label='Rejected', fill=True, alpha=0.4, color='tomato')

    # Mann-Whitney U test to check if distributions are statistically different
    u_stat, p_val = stats.mannwhitneyu(chosen[col].dropna(), rejected[col].dropna(), alternative='two-sided')
    ax.set_title(f'{label}  (p={p_val:.2e})')
    ax.set_xlabel(label)
    ax.legend()

plt.suptitle('Feature distributions: chosen vs rejected responses', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../../reports/figures/02_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Correlation heatmap

Which features move together? Highly correlated features don't add independent information, which matters for the logistic regression baseline.

In [ ]:
corr_cols = feature_cols + ['label']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title('Feature correlation matrix')
plt.tight_layout()
plt.savefig('../../reports/figures/03_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature-label correlation summary

Which features correlate most with the preference label? A quick ranking before we hand things off to the models.

In [ ]:
label_corr = df[corr_cols].corr()['label'].drop('label').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if v > 0 else 'tomato' for v in label_corr.values]
ax.barh(label_corr.index, label_corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature correlation with preference label')
ax.set_xlabel('Pearson correlation')
plt.tight_layout()
plt.savefig('../../reports/figures/04_label_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print(label_corr.round(4).to_string())

## Subset comparison: helpful-base vs harmless-base

The two subsets have different goals. Do they look different in feature space? If the feature distributions shift a lot between subsets, we may want to account for that during training.

In [ ]:
fig, axes = plt.subplots(1, len(feature_cols), figsize=(16, 5))

for i, (col, label) in enumerate(zip(feature_cols, feature_labels)):
    ax = axes[i]
    for subset, color in [('helpful-base', 'steelblue'), ('harmless-base', 'darkorange')]:
        vals = df[df['subset'] == subset][col]
        lo, hi = vals.quantile(0.01), vals.quantile(0.99)
        sns.kdeplot(vals.clip(lo, hi), ax=ax, label=subset, fill=True, alpha=0.35, color=color)
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.suptitle('Feature distributions by subset', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../../reports/figures/05_subset_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## UMAP: do the embeddings form natural clusters?

This is a preview of Week 2. We reduce 384-dimensional embeddings to 2D with UMAP and color by label. If chosen and rejected responses cluster separately, the embedding space has real signal. If they're tangled together, life gets harder.

Running on 5000 samples to keep it fast. The full clustering analysis comes in notebook 02.

In [ ]:
import umap

# subsample for speed
n_sample = 5000
idx = np.random.RandomState(42).choice(len(df), n_sample, replace=False)
emb_sample = embeddings[idx]
labels_sample = df['label'].values[idx]
subsets_sample = df['subset'].values[idx]

print(f'Running UMAP on {n_sample} samples...')
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
emb_2d = reducer.fit_transform(emb_sample)
print('Done.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# colored by preference label
scatter_colors = ['tomato' if l == 0 else 'steelblue' for l in labels_sample]
axes[0].scatter(emb_2d[:, 0], emb_2d[:, 1], c=scatter_colors, s=4, alpha=0.5)
axes[0].set_title('UMAP colored by label (blue = chosen, red = rejected)')
axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')

# colored by subset
subset_colors = ['darkorange' if s == 'harmless-base' else 'mediumseagreen' for s in subsets_sample]
axes[1].scatter(emb_2d[:, 0], emb_2d[:, 1], c=subset_colors, s=4, alpha=0.5)
axes[1].set_title('UMAP colored by subset (green = helpful, orange = harmless)')
axes[1].set_xlabel('UMAP 1')

plt.suptitle('UMAP projection of sentence embeddings (5K sample)', fontsize=13)
plt.tight_layout()
plt.savefig('../../reports/figures/06_umap_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

By now you should have a clear picture of the feature landscape. Key things to look for:

- If token_length or flesch_score shows a big shift between chosen and rejected, those features will be strong signals for the logistic regression baseline.
- If the UMAP plot shows blended classes rather than clear separation, that tells us engineered features alone may not be enough and the DeBERTa model will earn its complexity.
- If helpful-base and harmless-base have notably different distributions, consider tracking subset as a feature or stratifying the split.

Next up: formal clustering in the `02_clustering/` notebooks.